<a href="https://colab.research.google.com/github/thalbl/real-time-translator/blob/main/ml_final_project_t.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Real-Time Translator with AI

Academic project: **STT (Whisper) -> Translation (M2M100) -> TTS (SpeechT5)**
Adapted for Google Colab with browser recording and interactive interface.

| Stage | Model | Purpose |
|-------|-------|---------|
| STT | OpenAI Whisper | Speech -> Text |
| Translation | Meta M2M100 | Text language A -> Text language B |
| TTS | Microsoft SpeechT5 | Text -> Synthesized speech |

In [2]:
#C2
!pip install -q torch transformers sentencepiece scipy numpy

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"GPU detected {torch.cuda.get_device_name(0)}")
    print(f" VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU! Runtime → Change runtime type → T4 GPU")


No GPU! Runtime → Change runtime type → T4 GPU


In [1]:
# C3 - Clean up old models from memory
import gc
try: del stt
except: pass
try: del translator
except: pass
try: del tts
except: pass
gc.collect()
torch.cuda.empty_cache()
print(f"Memory cleared! Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

NameError: name 'torch' is not defined

In [ ]:
# ============================================================
# C4 - Imports
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import os, base64, tempfile, subprocess
import numpy as np
import torch
from IPython.display import display, Audio as IPAudio, Javascript, HTML, clear_output
import ipywidgets as widgets
import scipy.io.wavfile as wavfile

from transformers import (
    pipeline,
    M2M100ForConditionalGeneration,
    M2M100Tokenizer,
    SpeechT5ForTextToSpeech,
    SpeechT5HifiGan,
    SpeechT5Processor,
)

from google.colab import output as colab_output

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE.upper()}")

In [ ]:
# ============================================================
# C9 - Load all AI models
# ============================================================
print("Loading models... this may take a few minutes.\n")

# Re-initializing models to ensure they use the latest class definitions.
stt = STTModule(model_name="openai/whisper-medium", device=DEVICE)
print()

translator = TranslatorModule(source_lang="pt", target_lang="en", model_name="facebook/m2m100_1.2B", device=DEVICE)

print()

tts = TTSModule(device=DEVICE)

print("\n" + "=" * 50)
print("ALL MODELS LOADED SUCCESSFULLY!")
print("=" * 50)